<a href="https://colab.research.google.com/github/WaldaTzal2/gh0/blob/main/atividade_mlops_sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Clona o repositório da aula
!git clone https://github.com/profdiegoluispires/sentiment-analysis.git
%cd sentiment-analysis

# Instala as dependências necessárias
!pip install -q mlflow transformers fastapi uvicorn pydantic gradio nest-asyncio

Cloning into 'sentiment-analysis'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 28 (delta 7), reused 26 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (28/28), 11.70 KiB | 5.85 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/sentiment-analysis
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 50.6 MB/s eta 0:00:00

In [ ]:
import gradio as gr
from transformers import pipeline

# Carrega o modelo de sentimento
classifier = pipeline("sentiment-analysis")

def analyze_sentiment(text):
    result = classifier(text)[0]
    return f"Sentimento: {result['label']} | Confiança: {result['score']:.4f}"

# Cria e executa a interface com link público
demo = gr.Interface(
    fn=analyze_sentiment,
    inputs=gr.Textbox(lines=2, placeholder="Digite uma frase em inglês..."),
    outputs="text",
    title="Análise de Sentimento - Gradio"
)

demo.launch(share=True)

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://91f5c4d5995776665a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import mlflow
import time
from transformers import pipeline
import pandas as pd

# Define o experimento
mlflow.set_experiment("sentiment-analysis-colab")

# 3 modelos a serem comparados (adicionado o twitter-roberta)
MODELS_TO_COMPARE = [
    "distilbert-base-uncased-finetuned-sst-2-english",
    "finiteautomata/bertweet-base-sentiment-analysis",
    "cardiffnlp/twitter-roberta-base-sentiment-latest"
]

test_sentences = [
    "I absolutely love this product!",
    "This was the worst service I have ever received.",
    "It is an average movie, nothing special."
]

for model_name in MODELS_TO_COMPARE:
    with mlflow.start_run(run_name=model_name):
        start_time = time.time()
        pipe = pipeline("sentiment-analysis", model=model_name)

        # Inferência e métricas
        results = pipe(test_sentences)
        avg_confidence = sum([r['score'] for r in results]) / len(results)
        latency = (time.time() - start_time) / len(test_sentences)

        # Log de parâmetros e métricas no MLflow
        mlflow.log_param("model_name", model_name)
        mlflow.log_metric("avg_confidence", avg_confidence)
        mlflow.log_metric("latency_seconds", latency)

# Exibe a tabela comparativa de execuções
runs = mlflow.search_runs(experiment_names=["sentiment-analysis-colab"])
display(runs[["tags.mlflow.runName", "params.model_name", "metrics.avg_confidence", "metrics.latency_seconds"]])

2026/09/01 19:15:29 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/01 19:15:29 INFO mlflow.store.db.utils: Updating database tables
2026/09/01 19:15:32 INFO mlflow.tracking.fluent: Experiment with name 'sentiment-analysis-colab' does not exist. Creating a new experiment.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  501MB            

model.safetensors: downloading bytes:           |  0.00B            

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

,tags.mlflow.runName,params.model_name,metrics.avg_confidence,metrics.latency_seconds
0,cardiffnlp/twitter-roberta-base-sentiment-latest,cardiffnlp/twitter-roberta-base-sentiment-latest,0.909556,5.019708
1,finiteautomata/bertweet-base-sentiment-analysis,finiteautomata/bertweet-base-sentiment-analysis,0.970422,2.930418
2,distilbert-base-uncased-finetuned-sst-2-english,distilbert-base-uncased-finetuned-sst-2-english,0.999760,1.541891


In [ ]:
import nest_asyncio
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List
from transformers import pipeline

# Permite rodar o servidor assíncrono dentro do Colab
nest_asyncio.apply()

app = FastAPI(title="Sentiment Analysis API")
sentiment_pipeline = pipeline("sentiment-analysis")

class BatchInput(BaseModel):
    texts: List[str]

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict")
def predict(dado: dict):
    result = sentiment_pipeline(dado.get("texto", ""))
    return {"resultado": result}

# Implementação do endpoint em lote (Exercício B)
@app.post("/predict/batch")
def predict_batch(payload: BatchInput):
    results = sentiment_pipeline(payload.texts)
    return {"resultados": results}

# Teste direto do endpoint via cliente de teste
from fastapi.testclient import TestClient
client = TestClient(app)

# Executando a requisição de teste
response = client.post("/predict/batch", json={
    "texts": ["This is great!", "I hate waiting in line."]
})

print("Status Code:", response.status_code)
print("Resposta da API:", response.json())

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Status Code: 200
Resposta da API: {'resultados': [{'label': 'POSITIVE', 'score': 0.9998694658279419}, {'label': 'NEGATIVE', 'score': 0.9975144863128662}]}
